# OFR FSI 官方修订幅度与当前快照对账

只读取已归档官方 CSV 和修订工作簿；记录旧/新值差异，不构造市场特征。

In [1]:
import csv,json,hashlib,re,zipfile,xml.etree.ElementTree as ET
from pathlib import Path
from datetime import datetime,timedelta,date
import numpy as np
root=Path.cwd();card=root/'research/stage1_ofr_financial_stress_source_20261023';plan=json.loads((card/'revision_audit_plan.json').read_text())
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
raw=root/'data/snapshots/stage1_ofr_financial_stress_source_20261023';assert plan['frozen'] and sha(card/'plan.json')==plan['source_plan_sha256'] and sha(card/'source_result.json')==plan['source_result_sha256']
assert sha(raw/'fsi.csv')==plan['CSV_sha256'] and sha(raw/'FSI_Revision_History_2023-06-27.xlsx')==plan['revision_workbook_sha256']
source=json.loads((card/'source_result.json').read_text());assert source['NAS_remote_and_scoped_restore_verified'] and source['status'].startswith('OFR_FSI_official_current_full_span')
with (raw/'fsi.csv').open(newline='',encoding='utf-8-sig') as f:csv_rows=list(csv.DictReader(f))
current={date.fromisoformat(r['Date'][:10]):r for r in csv_rows};assert len(current)==len(csv_rows)
ns={'x':'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}
with zipfile.ZipFile(raw/'FSI_Revision_History_2023-06-27.xlsx') as z:
 strings=[''.join(t.text or '' for t in si.findall('.//x:t',ns)) for si in ET.fromstring(z.read('xl/sharedStrings.xml')).findall('x:si',ns)]
 def read_sheet(name):
  result=[];tree=ET.fromstring(z.read('xl/worksheets/'+name))
  for row in tree.findall('.//x:sheetData/x:row',ns):
   vals={}
   for cell in row.findall('x:c',ns):
    v=cell.find('x:v',ns)
    if v is None:continue
    value=v.text
    if cell.attrib.get('t')=='s':value=strings[int(value)]
    col=re.match(r'[A-Z]+',cell.attrib['r']).group(0);vals[col]=value
   if vals:result.append(vals)
  return result
 posted=[]
 for row in read_sheet('sheet1.xml'):
  try:
   when=(datetime(1899,12,30)+timedelta(days=float(row['A']))).date();desc=row['B']
   if 2017<=when.year<=2024 and len(desc)>30:posted.append({'posted_revision_date':str(when),'description':desc})
  except (KeyError,ValueError,TypeError):pass
 audits=[]
 for revision,sheet in plan['worksheets'].items():
  rows=read_sheet(sheet);pairs=[]
  for row in rows:
   try:
    d=(datetime(1899,12,30)+timedelta(days=float(row['A']))).date()
    if sheet=='sheet4.xml':
     assert float(row['A'])==float(row['K']);old_total,new_total=float(row['B']),float(row['L']);old_funding,new_funding=float(row['F']),float(row['P'])
    else:old_total,new_total=float(row['B']),float(row['C']);old_funding,new_funding=float(row['J']),float(row['K'])
    pairs.append({'date':d,'old_total':old_total,'updated_total':new_total,'old_funding':old_funding,'updated_funding':new_funding})
   except (KeyError,ValueError,TypeError):continue
  assert pairs and len({p['date'] for p in pairs})==len(pairs)
  diff_total=np.array([abs(p['updated_total']-p['old_total']) for p in pairs]);diff_funding=np.array([abs(p['updated_funding']-p['old_funding']) for p in pairs])
  common=[p for p in pairs if p['date'] in current];assert len(common)==len(pairs)
  snapshot_total=np.array([abs(p['updated_total']-float(current[p['date']]['OFR FSI'])) for p in common]);snapshot_funding=np.array([abs(p['updated_funding']-float(current[p['date']]['Funding'])) for p in common])
  audits.append({'revision_date':revision,'sheet':sheet,'paired_rows':len(pairs),'first_affected_date':str(min(p['date'] for p in pairs)),'last_affected_date':str(max(p['date'] for p in pairs)),'old_vs_updated_total_abs_median':float(np.median(diff_total)),'old_vs_updated_total_abs_max':float(diff_total.max()),'old_vs_updated_total_share_gt_0p1':float(np.mean(diff_total>0.1)),'old_vs_updated_funding_abs_median':float(np.median(diff_funding)),'old_vs_updated_funding_abs_max':float(diff_funding.max()),'old_vs_updated_funding_share_gt_0p1':float(np.mean(diff_funding>0.1)),'updated_vs_current_CSV_total_abs_max':float(snapshot_total.max()),'updated_vs_current_CSV_total_rows_gt_0p01':int(np.sum(snapshot_total>0.01)),'updated_vs_current_CSV_funding_abs_max':float(snapshot_funding.max()),'updated_vs_current_CSV_funding_rows_gt_0p01':int(np.sum(snapshot_funding>0.01))})
result={'schema_version':1,'status':'official_OFR_revision_materiality_audited_current_snapshot_not_PIT','plan_sha256':sha(card/'revision_audit_plan.json'),'source_NAS_archive_sha256':source['NAS_archive_sha256'],'posted_revision_events':posted,'paired_revision_sheets':audits,'current_csv_rows':len(current),'historical_first_public_vintage_qualified':False,'historical_methodology_revisions_fully_qualified':False,'SOXX_outcome_reads':0,'model_fits':0,'accounts':0,'interpretation':'The official revision workbook exposes original versus updated values for named 2018 and 2023 revision windows. The current CSV is a retrospective snapshot; posted revision list and two-business-day current page note do not certify all daily first-public versions, and the page reports an additional 2026 correction after workbook date.'}
(card/'revision_audit_result.json').write_text(json.dumps(result,indent=2)+'\n');print({'status':result['status'],'posted':posted,'audits':audits})


{'status': 'official_OFR_revision_materiality_audited_current_snapshot_not_PIT', 'posted': [{'posted_revision_date': '2018-05-18', 'description': 'Values of the OFR FSI and all of its subcomponents between 10/2/2017 and 5/17/2018 (inclusive) were revised to correct for missing values for the ICE BAML Euro Area Corp Bond Index (OAS), which was found not to be updating.'}, {'posted_revision_date': '2018-12-04', 'description': 'Values of the OFR FSI and all of its subcomponents on 8/30/2018 were revised to correct for erroneous input data received on that date for the MSCI Emerging Markets Index. Values of the OFR FSI and all of its subcomponents on 11/1/2018 were revised to correct for erroneous input data received on that date for the JPMorgan CEMBI Strip Spread and the JPMorgan EMBI Global Strip Spread.'}, {'posted_revision_date': '2021-02-12', 'description': 'The the JP Morgan Emerging Market Volatility Index (EM-VXY) was miscategorized. It is now properly classified as Emerging Marke

# OFR 金融压力指数：官方全历史来源与修订边界

官方 CSV 收到 6,784 个无重复日期（2000-01-03 至 2026-10-20），`Funding` 等九列均有数值；官方修订工作簿与 CSV 原字节已归档至 NAS，远端哈希及每个原始成员限定恢复通过。0 SOXX 结果读取、0 拟合、0 账户。

来源风险是实质性的：2018-05-18 的 158 个改值日期、2018-12-04 的 2 个日期，以及 2023-06-27 的 382 个日期均有官方旧/新值；2023 年替换资金类别变量后，Funding 绝对变动中位数 0.307、最大 0.764 个指数点。工作簿“新值”与当前 CSV 在对应日期逐项一致，不能把它当作当年旧值。官网另提示 2026 年数据订正，旧版本不在 2023 工作簿内。

[OFR 于 2017-10-25 发布的说明](https://www.financialresearch.gov/from-the-director/2017/10/25/the-ofr-introduces-financial-stability-index-and-financial-vulnerabilities-monitor/)把现行每日 FSI 列为新工具；[2014 年 OFR 报告](https://www.financialresearch.gov/annual-reports/files/office-of-financial-research-annual-report-2014.pdf)描述较早的 FSI，但不能证明现行 33 指标序列的 2000–2017 回填与当时版本完全相同。因此**不能**把今天 CSV 的 2011–2017 数值直接并入冻结 Stage1 全训练期当点时信号。

[当前 OFR 页面](https://www.financialresearch.gov/financial-stress-index/)称公布值较观测日滞后两个工作日；这不是各历史日的首次接收证明。下一步只可对 2018–2022 的 10:32 恢复接收器做单独的条件性来源映射，保守滞后、复原官方已列修订旧值、保留未知；通过后还须检验旧 21 输入增量和完整 0bp 账户。

来源 NAS SHA-256 `074e2c56f6f95a4f56382a32e25c65ad560e5a53ee6e4bb7fd58e50bc247b369`。本卡决定：`official_full_numeric_history_received_pre2017_PIT_and_revision_boundaries_unqualified`，不授予交易时点、SOXX 超额或晋升资格。